# INSTRUCTION-LEVEL PARALLELISM SIMULATOR (TAHAP 3: FULL MULTI-HAZARD)
**Kontributor / Komit:** Anggota 3 (Kelompok 4)
**Mata Kuliah:** Organisasi dan Arsitektur Komputer
**Repositori GitHub:** https://github.com/aarrrryyaaaaa/K4_Organisasi-Arsitektur-Komputer.git

---
### Deskripsi Tahap 3
Notebook ini merupakan integrasi lengkap ketiga data hazard arsitektur komputer:
1. **RAW** (*Read After Write* / *True Dependency*): $Dest(I_i) \cap Src(I_j) \neq \emptyset$
2. **WAR** (*Write After Read* / *Anti-Dependency*): $Src(I_i) \cap Dest(I_j) \neq \emptyset$
3. **WAW** (*Write After Write* / *Output Dependency*): $Dest(I_i) == Dest(I_j)$
4. Menghasilkan visualisasi **Full Topological Dependency Graph** dengan tingkat eksekusi Cycle 1 sampai Cycle 4.


In [ ]:
from dataclasses import dataclass
import matplotlib.pyplot as plt
import matplotlib.patches as patches

@dataclass
class Instruction:
    name: str
    destination: str
    sources: list

# Set Instruksi Komprehensif Mengandung RAW, WAR, dan WAW
instructions = [
    Instruction('I1', 'R1', ['R2', 'R3']),   # Menulis R1
    Instruction('I2', 'R4', ['R1', 'R5']),   # Membaca R1 (RAW thd I1), Menulis R4
    Instruction('I3', 'R1', ['R6', 'R7']),   # Menulis R1 (WAR thd I2 membaca R1, WAW thd I1 menulis R1)
    Instruction('I4', 'R4', ['R8', 'R9']),   # Menulis R4 (WAW thd I2 menulis R4)
    Instruction('I5', 'R10', ['R1', 'R4']),  # Membaca R1 dan R4 (RAW thd I3 dan I4)
    Instruction('I6', 'R12', ['R13', 'R14']) # Independen
]

# -------------------------------------------------
# Deteksi Ketiga Data Hazard (RAW, WAR, WAW)
# -------------------------------------------------
def check_raw(inst1, inst2):
    return inst1.destination in inst2.sources

def check_war(inst1, inst2):
    return inst2.destination in inst1.sources

def check_waw(inst1, inst2):
    return inst1.destination == inst2.destination

def has_dependency(inst1, inst2):
    return check_raw(inst1, inst2) or check_war(inst1, inst2) or check_waw(inst1, inst2)

def show_dependencies(instructions):
    print('\n' + '=' * 60)
    print('FULL DEPENDENCY ANALYSIS (RAW, WAR, WAW)')
    print('=' * 60)
    for i in range(len(instructions)):
        for j in range(i + 1, len(instructions)):
            u, v = instructions[i], instructions[j]
            if check_raw(u, v):
                print(f'[RAW] {v.name} membaca register {u.destination} dari {u.name}')
            if check_war(u, v):
                print(f'[WAR] {v.name} menulis register {v.destination} yang dibaca oleh {u.name}')
            if check_waw(u, v):
                print(f'[WAW] {v.name} menulis register {v.destination} sama seperti {u.name}')

# -------------------------------------------------
# Eksekusi Sekuensial
# -------------------------------------------------
def sequential_execution(instructions):
    print('\n' + '=' * 60)
    print('SEQUENTIAL EXECUTION')
    print('=' * 60)
    cycle = 0
    for inst in instructions:
        cycle += 1
        print(f'Cycle {cycle}: Executing {inst.name} ({inst.destination} = {" + ".join(inst.sources)})')
    print(f'\nTotal Sequential Cycles: {cycle}')
    return cycle

# -------------------------------------------------
# Eksekusi Paralel ILP (Mematuhi Seluruh Hazard)
# -------------------------------------------------
def parallel_execution(instructions):
    print('\n' + '=' * 60)
    print('PARALLEL ILP EXECUTION (RAW + WAR + WAW RESTRICTIONS)')
    print('=' * 60)
    completed = set()
    remaining = instructions.copy()
    cycle = 0

    while remaining:
        cycle += 1
        current_cycle = []

        for instruction in remaining:
            satisfied = True
            for prev in instructions:
                if prev.name == instruction.name:
                    break
                if has_dependency(prev, instruction):
                    if prev.name not in completed:
                        satisfied = False
                        break
            if satisfied:
                current_cycle.append(instruction)

        print(f'\nCycle {cycle}:')
        for inst in current_cycle:
            print(f'  -> {inst.name}: {inst.destination} = {" + ".join(inst.sources)}')

        for inst in current_cycle:
            completed.add(inst.name)
            remaining.remove(inst)

    print(f'\nTotal ILP Cycles: {cycle}')
    return cycle

# -------------------------------------------------
# Visualisasi Dependency Graph Lengkap (RAW, WAR, WAW)
# -------------------------------------------------
def draw_dependency_graph():
    fig, ax = plt.subplots(figsize=(10, 7), dpi=150)
    fig.patch.set_facecolor('#ffffff')
    ax.set_facecolor('#ffffff')
    levels = {
        'Level 1': (4.0, 'Eksekusi Cycle 1'),
        'Level 2': (2.7, 'Eksekusi Cycle 2 - Tunggu RAW R1'),
        'Level 3': (1.4, 'Eksekusi Cycle 3 - Tunggu WAR & WAW'),
        'Level 4': (0.1, 'Eksekusi Cycle 4 - Tunggu RAW Hasil Akhir')
    }
    for lvl, (y, desc) in levels.items():
        ax.axhline(y=y, color='#e2e8f0', linestyle='--', linewidth=1)
        ax.text(-0.2, y, f'{lvl}\n({desc})', fontsize=8, fontweight='bold', color='#64748b', ha='right', va='center')
    nodes = {
        'I1': {'pos': (1.5, 4.0), 'code': 'R1 = R2 + R3'},
        'I6': {'pos': (5.2, 4.0), 'code': 'R12 = R13 + R14'},
        'I2': {'pos': (1.5, 2.7), 'code': 'R4 = R1 + R5'},
        'I3': {'pos': (1.5, 1.4), 'code': 'R1 = R6 + R7'},
        'I4': {'pos': (5.2, 1.4), 'code': 'R4 = R8 + R9'},
        'I5': {'pos': (3.35, 0.1), 'code': 'R10 = R1 + R4'},
    }
    edges = [
        ('I1', 'I2', 'RAW (R1)', '#1d4ed8', '-'),
        ('I1', 'I3', 'WAW (R1)', '#6b21a8', '-.'),
        ('I2', 'I3', 'WAR (R1)', '#b45309', '--'),
        ('I2', 'I4', 'WAW (R4)', '#6b21a8', '-.'),
        ('I3', 'I5', 'RAW (R1)', '#1d4ed8', '-'),
        ('I4', 'I5', 'RAW (R4)', '#1d4ed8', '-'),
    ]
    for name, data in nodes.items():
        x, y = data['pos']
        rect = patches.FancyBboxPatch((x - 0.85, y - 0.35), 1.7, 0.7, boxstyle='round,pad=0.08,rounding_size=0.08',
                                      facecolor='#f8fafc', edgecolor='#1e293b', linewidth=1.5, zorder=3)
        ax.add_patch(rect)
        ax.text(x, y + 0.12, name, fontsize=11, fontweight='bold', ha='center', va='center', color='#0f172a', zorder=4)
        ax.text(x, y - 0.14, data['code'], fontsize=9.5, family='monospace', ha='center', va='center', color='#334155', zorder=4)
    for u, v, lbl, col, sty in edges:
        x1, y1 = nodes[u]['pos']
        x2, y2 = nodes[v]['pos']
        dx = x2 - x1
        dy = y2 - y1
        conn = 'arc3,rad=0.15' if abs(dx) > 0.1 else 'arc3,rad=0.0'
        if dx == 0 and abs(dy) > 1.5:
            conn = 'arc3,rad=0.25'
        arrow = patches.FancyArrowPatch((x1, y1 - 0.35), (x2, y2 + 0.35), connectionstyle=conn,
                                        arrowstyle='-|>', mutation_scale=15, color=col, linestyle=sty, linewidth=1.8, zorder=2)
        ax.add_patch(arrow)
        mid_x = (x1 + x2)/2 + (0.22 if dx == 0 else 0)
        mid_y = (y1 + y2)/2
        ax.text(mid_x, mid_y, f' {lbl} ', fontsize=8, fontweight='bold', color=col,
                bbox=dict(boxstyle='round,pad=0.2,rounding_size=0.1', facecolor='#ffffff', edgecolor=col, linewidth=1), ha='center', va='center', zorder=5)
    ax.set_xlim(-1.2, 7.2)
    ax.set_ylim(-0.8, 4.8)
    ax.axis('off')
    plt.title('Dependency Graph: Tahap 3 (Full Pipeline RAW, WAR, WAW)', fontsize=12, fontweight='bold', color='#0f172a')
    plt.tight_layout()
    plt.savefig('full_dependency_graph.png', dpi=300)
    plt.show()

# -------------------------------------------------
# Main Execution
# -------------------------------------------------
def main():
    print('*' * 60)
    print('   INSTRUCTION-LEVEL PARALLELISM SIMULATOR (RAW + WAR + WAW)')
    print('*' * 60)
    show_dependencies(instructions)
    seq_cycles = sequential_execution(instructions)
    ilp_cycles = parallel_execution(instructions)
    speedup = seq_cycles / ilp_cycles
    efficiency = (speedup / len(instructions)) * 100
    print('\n' + '=' * 60)
    print('PERFORMANCE COMPARISON')
    print('=' * 60)
    print(f'Sequential Cycles : {seq_cycles}')
    print(f'ILP Cycles        : {ilp_cycles}')
    print(f'Speedup           : {speedup:.2f}x')
    print(f'ILP Efficiency    : {efficiency:.2f}%')
    print('\nRendering Dependency Graph...')
    draw_dependency_graph()

if __name__ == '__main__':
    main()
